In [1]:
import fitz

In [2]:
import fitz
import json
import asyncio
import time
import sys
import re
import binascii
import unicodedata
import ollama
import nest_asyncio
import logging
from typing import List, Set, Optional, Dict
from hashlib import blake2b
from collections import Counter

# 1. INTENSIVE LOGGING SETUP
# Set to DEBUG to see the most granular information
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("DeepPipeline")

nest_asyncio.apply()

class TextGuardrails:
    def __init__(self, semantic_threshold: float = 0.85, num_perm: int = 64):
        self.threshold = semantic_threshold
        self.num_perm = num_perm
        self.exact_hashes: Set[str] = set()
        self.minhash_registry: List[List[int]] = []

    def is_duplicate(self, text: str) -> bool:
        h = blake2b(text.encode(), digest_size=16).hexdigest()
        if h in self.exact_hashes:
            return True
        
        if len(text) < 20: return False 
        
        # MinHash check
        clean = "".join(text.lower().split())
        shingles = {clean[i:i + 5] for i in range(len(clean) - 4)}
        if not shingles: return False
        
        current_sig = [min(binascii.crc32(f"{s}{j}".encode()) & 0xffffffff for s in shingles) for j in range(self.num_perm)]

        for existing_sig in self.minhash_registry:
            match_count = sum(a == b for a, b in zip(current_sig, existing_sig))
            if (match_count / self.num_perm) >= self.threshold:
                return True

        self.exact_hashes.add(h)
        self.minhash_registry.append(current_sig)
        return False

    def apply(self, text: str) -> Optional[str]:
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(r"[a-zA-Z0-9_.+-]+ email", "[EMAIL]", text) # Simplified PII for speed
        text = " ".join(text.split())
        
        sentences = re.split(r'(?<=[.!?]) +', text)
        unique_sentences = [s.strip() for s in sentences if s.strip() and not self.is_duplicate(s.strip())]
        
        result = " ".join(unique_sentences)
        return result if len(result) > 5 else None

class BatchPDFKeywordPipeline:
    def __init__(self, model_name="qwen2.5:7b", batch_size=6):
        self.model_name = model_name
        self.batch_size = batch_size
        self.semaphore = asyncio.Semaphore(2)

    async def process_batch(self, batch: List[Dict], batch_num: int):
        async with self.semaphore:
            logger.info(f"==> Batch {batch_num}: Sending {len(batch)} items to Ollama ({self.model_name})")
            try:
                prompt = (
                    "You are a Retail Banking Domain Expert. Return valid JSON only.\n"
                    "TASK: Assign EXACTLY 5 banking keywords per topic_id.\n"
                    f"DATA: {json.dumps(batch)}"
                )
                
                start_llm = time.time()
                response = await asyncio.wait_for(
                    asyncio.to_thread(
                        ollama.chat,
                        model=self.model_name,
                        format="json",
                        messages=[{"role": "user", "content": prompt}]
                    ), timeout=90.0
                )
                duration = time.time() - start_llm
                logger.info(f"==> Batch {batch_num}: Received LLM response in {duration:.2f}s")
                
                return json.loads(response["message"]["content"])
            except Exception as e:
                logger.error(f"!!! Batch {batch_num} FAILED: {str(e)}")
                return {}

    def extract_hierarchy(self, pdf_path: str) -> List[Dict]:
        logger.info(f"Step 1: Opening PDF at {pdf_path}")
        doc = fitz.open(pdf_path)
        spans = []
        
        logger.info(f"Step 1: Iterating through {len(doc)} pages...")
        for page_num, page in enumerate(doc, 1):
            blocks = page.get_text("dict")["blocks"]
            for b in blocks:
                if "lines" in b:
                    for l in b["lines"]:
                        for s in l["spans"]:
                            if s["text"].strip():
                                spans.append({"text": s["text"].strip(), "size": round(s["size"], 1)})
        
        logger.info(f"Step 1: Found {len(spans)} text spans. Analyzing font distribution...")
        size_counts = Counter([s["size"] for s in spans])
        unique_sizes = sorted(size_counts.keys(), reverse=True)
        
        logger.info(f"Step 1: Detected unique font sizes: {unique_sizes}")
        section_f = unique_sizes[0]
        topic_f = unique_sizes[1] if len(unique_sizes) > 1 else unique_sizes[0]
        logger.info(f"Step 1: Hierarchy Rule -> Section Size: {section_f}, Topic Size: {topic_f}")

        results, cur_sec_name, cur_sec_id = [], "Header", "S001"
        s_idx, t_idx = 1, 0

        for span in spans:
            if span["size"] == section_f and len(span["text"]) < 80:
                s_idx += 1
                cur_sec_name, cur_sec_id = span["text"], f"S{s_idx:03}"
                t_idx = 0
            elif span["size"] == topic_f and len(span["text"]) < 80:
                t_idx += 1
                results.append({
                    "section_id": cur_sec_id,
                    "section_name": cur_sec_name,
                    "topic_id": f"{cur_sec_id}_T{t_idx:03}",
                    "topic_name": span["text"],
                    "raw_lines": []
                })
            elif results:
                results[-1]["raw_lines"].append(span["text"])
        
        doc.close()
        logger.info(f"Step 1: Extraction complete. Identified {len(results)} potential topics.")
        return results

async def run_master_pipeline(pdf_path: str):
    overall_start = time.time()
    logger.info("PIPELINE INITIALIZED")
    
    pipeline = BatchPDFKeywordPipeline(batch_size=5)
    guardrails = TextGuardrails()

    # 1. Extraction
    raw_data = pipeline.extract_hierarchy(pdf_path)
    if not raw_data:
        logger.critical("No data found. Check PDF formatting or path.")
        return

    # 2. Cleaning
    logger.info("Step 2: Starting Text Cleaning and Deduplication...")
    processed_items = []
    for i, item in enumerate(raw_data):
        raw_string = " ".join(item.pop("raw_lines"))
        clean_text = guardrails.apply(raw_string)
        
        if clean_text:
            item["text"] = clean_text
            processed_items.append(item)
        
        if (i+1) % 20 == 0:
            logger.info(f"Step 2: Cleaned {i+1}/{len(raw_data)} items...")

    logger.info(f"Step 2: Finished. {len(processed_items)} items survived cleaning.")

    # 3. Keyword Generation
    total_items = len(processed_items)
    final_output = []
    logger.info(f"Step 3: Starting LLM Batch Processing (Batch Size: {pipeline.batch_size})")

    for i in range(0, total_items, pipeline.batch_size):
        batch_idx = (i // pipeline.batch_size) + 1
        batch = processed_items[i : i + pipeline.batch_size]
        
        # Progress Tracking
        progress = (i / total_items) * 100
        logger.info(f"Step 3: Progress: {progress:.1f}% | Handling items {i} to {min(i+pipeline.batch_size, total_items)}")
        
        llm_input = [{"topic_id": x["topic_id"], "text": x["text"][:600]} for x in batch]
        keywords_map = await pipeline.process_batch(llm_input, batch_idx)

        for item in batch:
            tid = item["topic_id"]
            item["keywords"] = keywords_map.get(tid, ["Banking", "Standard", "Retail"])
            if tid not in keywords_map:
                logger.warning(f"  -> Topic {tid} did not receive LLM keywords. Used defaults.")
            final_output.append(item)

    # 4. Save
    output_path = "structured_loan_data.json"
    logger.info(f"Step 4: Writing results to {output_path}")
    with open(output_path, "w") as f:
        json.dump(final_output, f, indent=4)
    
    total_duration = time.time() - overall_start
    logger.info(f"SUCCESS: Pipeline completed in {total_duration:.2f} seconds.")
    logger.info(f"Final Record Count: {len(final_output)}")

# RUN
PDF_FILE = r"F:\Loan_Details_RAG_System1\Data\loan_products_final.pdf"
await run_master_pipeline(PDF_FILE)

2026-04-13 11:53:53,826 | INFO | PIPELINE INITIALIZED
2026-04-13 11:53:53,826 | INFO | Step 1: Opening PDF at F:\Loan_Details_RAG_System1\Data\loan_products_final.pdf
2026-04-13 11:53:53,876 | INFO | Step 1: Iterating through 35 pages...
2026-04-13 11:53:54,049 | INFO | Step 1: Found 1832 text spans. Analyzing font distribution...
2026-04-13 11:53:54,052 | INFO | Step 1: Detected unique font sizes: [18.0, 13.4, 13.0, 12.5, 12.0, 10.1]
2026-04-13 11:53:54,057 | INFO | Step 1: Hierarchy Rule -> Section Size: 18.0, Topic Size: 13.4
2026-04-13 11:53:54,068 | INFO | Step 1: Extraction complete. Identified 138 potential topics.
2026-04-13 11:53:54,075 | INFO | Step 2: Starting Text Cleaning and Deduplication...
2026-04-13 11:53:55,137 | INFO | Step 2: Cleaned 20/138 items...
2026-04-13 11:53:56,150 | INFO | Step 2: Cleaned 40/138 items...
2026-04-13 11:53:57,036 | INFO | Step 2: Cleaned 60/138 items...
2026-04-13 11:53:58,420 | INFO | Step 2: Cleaned 80/138 items...
2026-04-13 11:53:59,080 |

In [3]:
with open("structured_loan_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [12]:
text = unicodedata.normalize("NFKC", data[4]["text"])

text

'The margin depends on the residual maturity period of the security: \uf0b7 NSC / KVP / Government Bonds o 15% of face value if residual maturity is less than 3 years o 20% of face value if residual maturity is 3 years or more \uf0b7 Life Insurance Policies o 15% of surrender value if maturity is within 3 years o 20% of surrender value if maturity is 3 years or more For staff borrowers against NSC, a concessional margin of 10% of face value applies.'

In [1]:
print(5+2)

7


In [4]:
output = {'outline information regarding automobile financing': ['DriveSure Auto Loan (DAL)'],
 'detail options for vehicle loan products': ['DriveSure Auto Loan (DAL)']}

In [5]:
import asyncio
import sys
import time
from typing import Dict, List, Any
from src.RAG.Strategies_RAG_Inference.retriever import HybridRetriever

async def main() -> Dict[str, List[Dict[str, Any]]]:
    """
    Production-grade entry point for the Hybrid Retrieval pipeline.
    Returns the final retrieved chunks for all queries in the batch.
    """
    
    # 1. Mock Input Data (The 'output' map provided)
    query_section_map = {
        'outline information regarding automobile financing': ['DriveSure Auto Loan (DAL)'], 
        'detail options for vehicle loan products': ['DriveSure Auto Loan (DAL)']
    }

    try:
        # 2. Initialize Retriever
        retriever = HybridRetriever()

        print("\n" + "="*70)
        print(" 🚀 STARTING HYBRID RETRIEVAL BATCH PROCESS")
        print("="*70)
        
        start_time = time.perf_counter()

        # 3. Execute Batch Retrieval
        # Flow: Filter-First -> Deduplicate -> Hybrid Search
        results = await retriever.retrieve_batch(
            query_section_map, 
            vector_k=5, 
            bm25_k=5
        )

        total_time = time.perf_counter() - start_time

        # 4. Result Presentation
        if not results:
            print("[!] No results returned from the pipeline.")
            return {}

        for query, docs in results.items():
            print(f"\nQUERY: {query}")
            print(f"SECTIONS SEARCHED: {query_section_map.get(query)}")
            print("-" * 70)

            if not docs:
                print("  ⚠️  No relevant documents found in the specified sections.")
                continue

            for i, doc in enumerate(docs):
                doc_id   = doc.get('id', 'N/A')
                section  = doc.get('metadata', {}).get('section_name', 'Unknown')
                source   = doc.get('retrieval_source', 'UNKNOWN').upper()
                score    = doc.get('score', 0.0)
                text_raw = doc.get('text', 'No text content available')
                preview  = " ".join(text_raw.split())[:120]

                print(f" [{i+1}] [{source:<6}] ID: {doc_id:<5} | Score: {score:.4f}")
                print(f"     Section: {section}")
                print(f"     Preview: {preview}...")
            
            print("-" * 70)

        print(f"\n✅ Processed {len(results)} queries in {total_time:.2f} seconds.")
        
        # 5. Return the final data payload
        return results

    except Exception as e:
        print(f"\n[CRITICAL ERROR] Pipeline failed: {str(e)}")
        return {}

if __name__ == "__main__":
    try:
        # Capture the returned chunks from the event loop
        final_chunks = await main()
        
        # Example of handling the returned output outside the loop
        if final_chunks:
            total_count = sum(len(chunks) for chunks in final_chunks.values())
            print(f"\n[OUTPUT] Successfully returned {total_count} total chunks to the caller.")
            
    except KeyboardInterrupt:
        print("\n[!] Process manually interrupted. Exiting...")
        sys.exit(0)

[2026-04-15 21:17:03,251] [INFO] [INIT] Initializing FAISS HNSW | dim=768, M=32
[2026-04-15 21:17:03,251] [INFO] [INIT COMPLETE] FAISS ready

 🚀 STARTING HYBRID RETRIEVAL BATCH PROCESS
[2026-04-15 21:17:03,251] [INFO] [SYNC] Starting incremental DB sync
2026-04-15 21:17:03,453 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-15 21:17:03,454 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-15 21:17:03,490 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-15 21:17:03,490 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-15 21:17:03,504 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-15 21:17:03,506 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-15 21:17:03,513 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-15 21:17:03,525 INFO sqlalchemy.engine.Engine SELECT document_chunks.id, document_chunks.chunk_text, document_chunks.embedding, document_chunks.confidence_score, document_chunks.chunk_metadata, document_chunks.prev_chun

In [6]:
final_chunks

{'outline information regarding automobile financing': [{'id': 155,
   'chunk_text': 'DriveSure Auto Loan provides a structured, transparent, and risk-controlled vehicle financing',
   'metadata': {'text': 'DriveSure Auto Loan provides a structured, transparent, and risk-controlled vehicle financing solution that balances customer affordability , operational discipline , and preventive vigilance . The product enables borrowers to acquire personal vehicles with flexible repayment options while ensuring strong asset protection and regulatory compliance.',
    'keywords': ['drivesure', 'auto', 'loan', 'provides', 'structured'],
    'topic_id': 'S003_T018',
    'section_id': 'S003',
    'topic_name': 'Summary',
    'section_name': 'DriveSure Auto Loan (DAL)',
    'index_in_para': 1},
   'confidence': 0.94,
   'context_ids': [],
   'distance': 262.7311706542969,
   'is_context': False,
   'retrieval_source': 'vector'},
  {'id': 82,
   'chunk_text': 'The loan may be availed for the purchase 

In [7]:
import sys
import ollama
from typing import List, Any
from src.Utils.logger_setup import setup_logger, track_performance
from src.Utils.exception_handler import CustomException

logger = setup_logger("generator_func")

@track_performance
async def generate_response(
    queries: List[str], 
    retrieved_chunks: List[Any], 
    ltm: str = "User is inquiring about automotive financial services and DriveSure products.", 
    stm: str = "Focusing on DriveSure Auto Loan (DAL) specific features."
) -> str:
    """
    Standalone RAG generation function.
    
    Args:
        queries: List of query strings (the keys from your output map).
        retrieved_chunks: List of dictionaries or objects containing 'chunk_text'.
        ltm: Long-term memory (mocked by default).
        stm: Short-term memory (mocked by default).
    """
    try:
        # 1. Merge Queries with a "."
        merged_query = ". ".join(queries) if queries else "No query provided."

        # 2. Extract Text from Chunks
        # Logic handles both objects (chunk.chunk_text) and dicts (chunk['chunk_text'])
        primary_list = []
        context_list = []

        for chunk in retrieved_chunks:
            # Flexible extraction for different data types
            if isinstance(chunk, dict):
                text = chunk.get("chunk_text", "")
                source = chunk.get("retrieval_source", "unknown").lower()
                is_context = chunk.get("is_context", False)
            else:
                text = getattr(chunk, "chunk_text", "")
                source = getattr(chunk, "retrieval_source", "unknown").lower()
                is_context = getattr(chunk, "is_context", False)

            text_block = f"[{source.upper()}] {text}"

            # Sort into Primary or Supporting based on source/type
            if "context" in source or is_context:
                context_list.append(text_block)
            else:
                primary_list.append(text_block)

        primary_text = "\n".join(primary_list) if primary_list else "No direct data available."
        context_text = "\n".join(context_list) if context_list else "No additional background."

        # 3. Construct the RAG Prompt
        full_prompt = f"""
        SYSTEM ROLE: You are a precise technical assistant for automobile financing. 
        
        ### PRIMARY REFERENCE DATA:
        {primary_text}

        ### SUPPORTING BACKGROUND INFORMATION:
        {context_text}

        ### CONVERSATION MEMORY:
        - LTM: {ltm}
        - STM: {stm}

        ### USER QUESTION:
        {merged_query}

        ### INSTRUCTIONS:
        1. Answer the question using the PRIMARY REFERENCE DATA.
        2. Use SUPPORTING BACKGROUND only if the primary data is insufficient.
        3. Maintain continuity with the provided CONVERSATION MEMORY.
        4. If the data provided does not contain the answer, state that you do not have that information.
        """

        # 4. Invoke Ollama (Ensure 'phi3.5' or 'qwen2.5' is pulled)
        response = ollama.chat(
            model="qwen2.5:7b",
            messages=[{'role': 'user', 'content': full_prompt}],
            options={'temperature': 0.1}
        )

        return response.get('message', {}).get('content', "No response generated.")

    except Exception as e:
        logger.error(f"Generation Function Error: {str(e)}")
        raise CustomException(e, sys)

In [12]:
output

{'outline information regarding automobile financing': ['DriveSure Auto Loan (DAL)'],
 'detail options for vehicle loan products': ['DriveSure Auto Loan (DAL)']}

In [13]:
final_chunks 

{'outline information regarding automobile financing': [{'id': 155,
   'chunk_text': 'DriveSure Auto Loan provides a structured, transparent, and risk-controlled vehicle financing',
   'metadata': {'text': 'DriveSure Auto Loan provides a structured, transparent, and risk-controlled vehicle financing solution that balances customer affordability , operational discipline , and preventive vigilance . The product enables borrowers to acquire personal vehicles with flexible repayment options while ensuring strong asset protection and regulatory compliance.',
    'keywords': ['drivesure', 'auto', 'loan', 'provides', 'structured'],
    'topic_id': 'S003_T018',
    'section_id': 'S003',
    'topic_name': 'Summary',
    'section_name': 'DriveSure Auto Loan (DAL)',
    'index_in_para': 1},
   'confidence': 0.94,
   'context_ids': [],
   'distance': 262.7311706542969,
   'is_context': False,
   'retrieval_source': 'vector'},
  {'id': 82,
   'chunk_text': 'The loan may be availed for the purchase 

In [9]:
import sys
import ollama
from typing import List, Any, Dict
from src.Utils.logger_setup import setup_logger, track_performance
from src.Utils.exception_handler import CustomException

logger = setup_logger("generator_func")

@track_performance
async def generate_response(
    query_dict: Dict[str, List[str]], 
    retrieved_chunks: List[Dict[str, Any]], 
    ltm: str = "User is inquiring about automotive financial services and DriveSure products.", 
    stm: str = "Focusing on DriveSure Auto Loan (DAL) specific features."
) -> str:
    """
    RAG generation function that merges dictionary keys for the query and
    categorizes chunks based on retrieval source.
    """
    try:
        # 1. Extract keys from query_dict and merge with "."
        queries = list(query_dict.keys())
        merged_query = ". ".join(queries) if queries else "No query provided."

        # 2. Extract and Categorize Chunks
        primary_list = []
        context_list = []

        for chunk in retrieved_chunks:
            # Get text and metadata
            text = chunk.get("chunk_text", "")
            source = str(chunk.get("retrieval_source", "unknown")).lower()
            is_context = chunk.get("is_context", False)

            # Format the source tag
            # If is_context is True, override source tag to CONTEXT
            source_tag = "CONTEXT" if is_context else source.upper()
            text_block = f"[{source_tag}] {text}"

            # Sort into Primary (Vector/BM25) or Supporting (Context)
            if is_context:
                context_list.append(text_block)
            else:
                primary_list.append(text_block)

        primary_text = "\n".join(primary_list) if primary_list else "No direct data available."
        context_text = "\n".join(context_list) if context_list else "No additional background."

        # 3. Construct the RAG Prompt
        full_prompt = f"""
        SYSTEM ROLE: You are a precise technical assistant for automobile financing. 
        
        ### PRIMARY REFERENCE DATA:
        {primary_text}

        ### SUPPORTING BACKGROUND INFORMATION:
        {context_text}

        ### CONVERSATION MEMORY:
        - LTM: {ltm}
        - STM: {stm}

        ### USER QUESTION:
        {merged_query}

        ### INSTRUCTIONS:
        1. Answer the question using the PRIMARY REFERENCE DATA.
        2. Use SUPPORTING BACKGROUND only if the primary data is insufficient.
        3. Maintain continuity with the provided CONVERSATION MEMORY.
        4. If the data provided does not contain the answer, state that you do not have that information.
        """

        # 4. Invoke Ollama
        response = ollama.chat(
            model="qwen2.5:7b",
            messages=[{'role': 'user', 'content': full_prompt}],
            options={'temperature': 0.1}
        )

        return response.get('message', {}).get('content', "No response generated.")

    except Exception as e:
        logger.error(f"Generation Function Error: {str(e)}")
        raise CustomException(e, sys)

In [ ]:


# 2. Run the generator directly using await
# In Jupyter, top-level await is supported.
try:
    # Extract the list of chunks from the first key's value
    chunks_list = list(final_chunks.values())[0]

    response = await generate_response(
        query_dict=output,
        retrieved_chunks=chunks_list,  # Pass the list, not the dict
        ltm="User is inquiring about automotive financial services.",
        stm="Focusing on DriveSure Auto Loan (DAL) specific features."
    )
    
    print("--- AI RESPONSE ---")
    print(response)
except Exception as e:
    print(f"Error: {e}")